# 🧪 Dokuz Model Protokolü — Makale-Uyumlu Segmentasyon Eğitimi (Colab, resumable)

Bu notebook, makaledeki **Tablo 13** (çok-mimarili karşılaştırma) için 9 mimariyi **aynı bölme, aynı ölçüt
ve aynı tohumlarla** eğitir. Sonuçlar Google Drive'a yazılır; tamamlanan (model, tohum) koşumları **atlanır**,
böylece oturum kopsa da kaldığı yerden devam eder.

### Protokol
- **Mimariler (9):** SegFormer MiT-B4/B2/B0 · U-Net / FPN / DeepLabV3+ / UPerNet (MiT-B4) · DeepLabV3+ ResNet-101 · UPerNet ConvNeXt-S
- **Tohum:** model başına **10 tohum** (karar 2026-09-20). `CONFIG['seeds']` ile ayarlanır
- **Kayıp:** Focal(α=0,25, γ=2) + Dice (0,5+0,5), `ignore_index=255`
- **Ölçüt:** IoU_hasar — **tüm sınama pikselleri üzerinden (global)** + kesinlik/duyarlılık/F1
- **Veri:** KATE-CD (resmî bölme + öncesi-görüntü negatifleri) + MEDBFS (sahne-bazlı bölme + öncesi negatifleri)

> ### ⚠️ Bölme kısıtı (dürüst not)
> HuggingFace `CSCRS/kate-cd` sürümünde karo koordinatı YOK → makalenin hedeflediği **2 km × 2 km UTM blok**
> bölmesi bu notebook'ta birebir yapılamıyor. Uygulanan: KATE-CD resmî train/val/test + MEDBFS sahne-bazlı
> bölme (sızıntıyı azaltır). Tam 2 km-blok bölmesi KATE-PD'nin CodeOcean ızgara dosyalarını gerektirir.

> ### ⏱️ Süre & resumability
> 9 model × 10 tohum = **90 eğitim** — tek oturum yetmez. Notebook Drive'a kaydeder ve tamamlananı atlar.
> Oturum kopunca **tekrar "Tümünü çalıştır"** de; kaldığı yerden devam eder. Döngü **tohum-dış, model-iç**:
> ilk tohum bitince 9 modelin tamamı için bir Tablo 13 (n=1) elde edilir.

**ÖNEMLİ:** `Çalışma Zamanı → türünü değiştir → T4 GPU` seçin.

In [ ]:
!pip install -q datasets opencv-python numpy tqdm scikit-learn torch torchvision segmentation-models-pytorch albumentations tifffile timm

## 1. Google Drive + Yapılandırma

In [ ]:
import os, re, numpy as np, cv2

# Google Drive (kalıcı kayıt + resume). Colab dışıysa yerel klasöre düşer.
DRIVE_ROOT = 'runs_dokuz_model'
try:
    from google.colab import drive
    drive.mount('/content/drive')
    DRIVE_ROOT = '/content/drive/MyDrive/QuakeMind_egitim/dokuz_model'
except Exception as e:
    print('Drive bağlanamadı (Colab dışı olabilir), yerel klasör kullanılacak:', e)
os.makedirs(DRIVE_ROOT, exist_ok=True)

IMG_DIR, MASK_DIR = 'data/images/', 'data/masks/'
for d in [IMG_DIR, MASK_DIR]:
    os.makedirs(d, exist_ok=True)

IGNORE_INDEX = 255  # maske: 0=arka plan, 1=hasar, 255=yok say

CONFIG = {
    'seeds':   [42, 7, 2026, 1, 13, 21, 34, 55, 89, 144],  # model başına 10 tohum (karar 2026-09-20)
    'epochs':  40,
    'patience': 8,
    'batch':   8,
    'lr':      2e-4,
    'focal_alpha': 0.25,   # A-19: 0.5/0.75 denenebilir; karşılaştırma için tüm modellerde aynı
    'focal_gamma': 2.0,
    'pretrained': 'imagenet',   # smoke-test için None yapılabilir
    'results_csv': os.path.join(DRIVE_ROOT, 'tablo13_sonuclar.csv'),
    'ckpt_dir': os.path.join(DRIVE_ROOT, 'ckpt'),
    'kate_neg': True, 'medbfs_neg': True,
}
os.makedirs(CONFIG['ckpt_dir'], exist_ok=True)

# 9 mimari: (etiket, smp mimari sınıfı, encoder adı)
MODELS = [
    ('segformer_mitb4',   'Segformer',     'mit_b4'),
    ('segformer_mitb2',   'Segformer',     'mit_b2'),
    ('segformer_mitb0',   'Segformer',     'mit_b0'),
    ('unet_mitb4',        'Unet',          'mit_b4'),
    ('fpn_mitb4',         'FPN',           'mit_b4'),
    ('deeplabv3p_mitb4',  'DeepLabV3Plus', 'mit_b4'),
    ('upernet_mitb4',     'UPerNet',       'mit_b4'),
    ('deeplabv3p_r101',   'DeepLabV3Plus', 'resnet101'),
    ('upernet_convnexts', 'UPerNet',       'tu-convnext_small'),
]

def make_name(source, scene, split, polarity, ident):
    return f'{source}__{str(scene)}__{split}__{polarity}__{str(ident)}.png'

def parse_name(fn):
    p = (fn[:-4] if fn.lower().endswith('.png') else fn).split('__')
    if len(p) < 5: return None
    return {'source':p[0],'scene':p[1],'split':p[2],'polarity':p[3],'ident':'__'.join(p[4:]),'file':fn}

print('Drive kökü:', DRIVE_ROOT)
print('Model sayısı:', len(MODELS), '| tohum sayısı:', len(CONFIG['seeds']),
      '| toplam koşum:', len(MODELS)*len(CONFIG['seeds']))

## 2. Veri — KATE-CD (resmî bölme + öncesi negatif)

In [ ]:
from datasets import load_dataset

def _to_bgr(pil):
    a = np.array(pil)
    if a.ndim == 3 and a.shape[2] == 3:
        a = cv2.cvtColor(a, cv2.COLOR_RGB2BGR)
    return a
def _save(img, mask, name):
    cv2.imwrite(os.path.join(IMG_DIR, name), img)
    cv2.imwrite(os.path.join(MASK_DIR, name), mask)

# Zaten indirilmişse atla (resume)
if any(f.startswith('kate__') for f in os.listdir(IMG_DIR)):
    print('KATE-CD zaten hazır, atlanıyor.')
else:
    print('KATE-CD çekiliyor (resmî train/validation/test)...')
    try:
        ds = load_dataset('CSCRS/kate-cd')
    except Exception as e:
        print('Alternatif:', e); ds = load_dataset('hkayabilisim/kate-cd')
    rng = np.random.default_rng(42); pos = neg = 0
    for split in ['train','validation','test']:
        if split not in ds: continue
        for i, v in enumerate(ds[split]):
            post = v.get('post_image') or v.get('image'); pre = v.get('pre_image'); lab = v.get('label') or v.get('mask')
            if post is None or lab is None: continue
            m = np.array(lab)
            if m.ndim > 2: m = cv2.cvtColor(m, cv2.COLOR_RGB2GRAY)
            m = np.where(m > 0, 1, 0).astype(np.uint8)
            pb = _to_bgr(post)
            if pb.shape[:2] == m.shape[:2]:
                _save(pb, m, make_name('kate','none',split,'pos',i)); pos += 1
            if CONFIG['kate_neg'] and pre is not None:
                prb = _to_bgr(pre); _save(prb, np.zeros(prb.shape[:2],np.uint8), make_name('kate','none',split,'neg',i)); neg += 1
    print(f'✅ KATE-CD: {pos} pozitif + {neg} negatif')

## 3. Veri — MEDBFS (sahne-bazlı bölme + öncesi negatif)

In [ ]:
import urllib.request, zipfile, tifffile
from tqdm import tqdm
from collections import Counter

if any(f.startswith('medbfs__') for f in os.listdir(IMG_DIR)):
    print('MEDBFS zaten hazır, atlanıyor.')
else:
    zip_path, extract = '/content/Turkey.zip', '/content/MEDBFS_Turkey'
    if not os.path.exists(zip_path):
        print('MEDBFS indiriliyor (~3.3GB)...')
        urllib.request.urlretrieve('https://zenodo.org/records/11647956/files/Turkey.zip?download=1', zip_path)
    if not os.path.exists(extract):
        print('Çıkarılıyor...')
        with zipfile.ZipFile(zip_path) as z: z.extractall(extract)
    post_dir = extract + '/Turkey/Image/Post-disaster'
    pre_dir  = extract + '/Turkey/Image/Pre-disaster'
    lbl_dir  = extract + '/Turkey/Label'

    def scene_origin(p):
        try:
            with tifffile.TiffFile(p) as t:
                tags = t.pages[0].tags
                tp = None
                for k in ('ModelTiepointTag', 33922):
                    try: tp = tags.get(k)
                    except Exception: tp = None
                    if tp is not None: break
                if tp is None: return None
                v = list(tp.value)
                if len(v) >= 6: return (round(float(v[3]),1), round(float(v[4]),1))
        except Exception: return None
        return None

    files = sorted(f for f in os.listdir(post_dir) if f.endswith('.tif') and os.path.exists(os.path.join(lbl_dir,f)))
    origins = {f: scene_origin(os.path.join(post_dir,f)) for f in tqdm(files, desc='sahne-orijini')}
    distinct = sorted({o for o in origins.values() if o is not None}); n = len(distinct)
    if 2 <= n <= 40:
        o2s = {o:f's{i}' for i,o in enumerate(distinct)}; fscene = {f:o2s.get(origins[f],'sX') for f in files}
        print(f'✅ {n} sahne saptandı; sahne-bazlı bölme.')
    else:
        fscene = {f:'s0' for f in files}
        print(f'⚠️ sahne-orijini gruplayamadı (n={n}) → tüm MEDBFS EĞİTİME (val/test KATE-CD\'den).')
    sc = Counter(fscene.values()); ssplit = {}
    if len(sc) >= 2:
        tgt = {'train':.8,'validation':.1,'test':.1}; asg = {'train':0,'validation':0,'test':0}; tot=sum(sc.values())
        for scene,c in sc.most_common():
            pick = max(tgt, key=lambda s: tgt[s]-asg[s]/max(tot,1)); ssplit[scene]=pick; asg[pick]+=c
    else:
        ssplit = {s:'train' for s in sc}
    for scene,c in sc.most_common(): print(f'   {scene}: {c} → {ssplit[scene]}')

    rng = np.random.default_rng(43); pos = neg = 0
    for f in tqdm(files, desc='MEDBFS kaydet'):
        scene = fscene[f]; split = ssplit[scene]
        img = cv2.imread(os.path.join(post_dir,f)); mask = cv2.imread(os.path.join(lbl_dir,f), cv2.IMREAD_GRAYSCALE)
        if img is None or mask is None or img.shape[:2] != mask.shape[:2]: continue
        mask = np.where(mask>0,1,0).astype(np.uint8); base = os.path.splitext(f)[0]
        _save(img, mask, make_name('medbfs',scene,split,'pos',base)); pos += 1
        pp = os.path.join(pre_dir,f)
        if CONFIG['medbfs_neg'] and os.path.exists(pp):
            pre = cv2.imread(pp)
            if pre is not None: _save(pre, np.zeros(pre.shape[:2],np.uint8), make_name('medbfs',scene,split,'neg',base)); neg += 1
    print(f'✅ MEDBFS: {pos} pozitif + {neg} negatif')

## 4. Manifest + bölme raporu

In [ ]:
from collections import defaultdict, Counter
manifest = [r for r in (parse_name(f) for f in sorted(os.listdir(IMG_DIR))) if r and os.path.exists(os.path.join(MASK_DIR, r['file']))]
by = defaultdict(Counter)
for r in manifest: by[r['split']][r['polarity']] += 1
print(f'{"bölme":<12}{"pozitif":>9}{"negatif":>9}{"toplam":>9}')
for s in ['train','validation','test']:
    c = by[s]; print(f'{s:<12}{c["pos"]:>9}{c["neg"]:>9}{c["pos"]+c["neg"]:>9}')
scsp = defaultdict(set)
for r in manifest:
    if r['source']=='medbfs': scsp[r['scene']].add(r['split'])
leaks = {k:sorted(v) for k,v in scsp.items() if len(v)>1}
print('Sızıntı:', leaks if leaks else 'YOK ✅', '| toplam karo:', len(manifest))

## 5. Ortak eğitim bileşenleri

In [ ]:
import torch, shutil, csv
from torch.utils.data import Dataset, DataLoader
import albumentations as A
from albumentations.pytorch import ToTensorV2
import segmentation_models_pytorch as smp
import torch.optim as optim

IMEAN, ISTD = [0.485,0.456,0.406], [0.229,0.224,0.225]
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

class DepremDataset(Dataset):
    def __init__(self, files, tf): self.files, self.tf = files, tf
    def __len__(self): return len(self.files)
    def __getitem__(self, i):
        n = self.files[i]
        img = cv2.cvtColor(cv2.imread(os.path.join(IMG_DIR,n)), cv2.COLOR_BGR2RGB)
        m = cv2.imread(os.path.join(MASK_DIR,n), cv2.IMREAD_GRAYSCALE).astype(np.float32)
        a = self.tf(image=img, mask=m)
        return a['image'], a['mask'].long()

train_tf = A.Compose([
    A.HorizontalFlip(p=.5), A.VerticalFlip(p=.5), A.RandomRotate90(p=.5),
    A.ShiftScaleRotate(shift_limit=.05, scale_limit=.1, rotate_limit=20, p=.5, border_mode=cv2.BORDER_REFLECT_101),
    A.RandomResizedCrop(size=(512,512), scale=(.7,1.), ratio=(.9,1.1), p=.3),
    A.ColorJitter(.3,.3,.2,.05, p=.5), A.CLAHE(4., (8,8), p=.2),
    A.GaussianBlur((3,5), p=.15), A.GaussNoise(std_range=(.02,.1), p=.15),
    A.Normalize(IMEAN, ISTD), ToTensorV2()])
val_tf = A.Compose([A.Normalize(IMEAN, ISTD), ToTensorV2()])

def build_model(arch, encoder):
    return getattr(smp, arch)(encoder_name=encoder, encoder_weights=CONFIG['pretrained'], in_channels=3, classes=1)

def make_loss():
    d = smp.losses.DiceLoss(mode='binary', ignore_index=IGNORE_INDEX)
    f = smp.losses.FocalLoss(mode='binary', alpha=CONFIG['focal_alpha'], gamma=CONFIG['focal_gamma'], ignore_index=IGNORE_INDEX)
    return lambda p,t: 0.5*d(p,t) + 0.5*f(p,t)

@torch.no_grad()
def metrics(model, loader, loss_fn):
    model.eval(); tp=fp=fn=0.0; tl=0.0; nb=0
    for img, mk in loader:
        img = img.to(device); m = mk.to(device).unsqueeze(1).float()
        out = model(img); tl += loss_fn(out,m).item(); nb += 1
        val = (m!=IGNORE_INDEX); tgt = (m==1)&val; pred = (torch.sigmoid(out)>0.5)&val
        tp += (pred&tgt).sum().item(); fp += (pred&~tgt&val).sum().item(); fn += (~pred&tgt).sum().item()
    iou = tp/(tp+fp+fn) if (tp+fp+fn)>0 else float('nan')
    prec = tp/(tp+fp) if (tp+fp)>0 else float('nan'); rec = tp/(tp+fn) if (tp+fn)>0 else float('nan')
    f1 = 2*prec*rec/(prec+rec) if (prec and rec and prec+rec>0) else float('nan')
    return tl/max(nb,1), iou, prec, rec, f1

def files_for(split): return [r['file'] for r in manifest if r['split']==split]

def train_one_run(label, arch, encoder, seed):
    torch.manual_seed(seed); np.random.seed(seed)
    tr = DataLoader(DepremDataset(files_for('train'), train_tf), batch_size=CONFIG['batch'], shuffle=True, num_workers=2, pin_memory=True, drop_last=True)  # son tek-örnekli batch BatchNorm'u (DeepLabV3+/UPerNet) çökertir
    va = DataLoader(DepremDataset(files_for('validation'), val_tf), batch_size=CONFIG['batch'], shuffle=False, num_workers=2, pin_memory=True)
    model = build_model(arch, encoder).to(device)
    loss_fn = make_loss()
    opt = optim.AdamW(model.parameters(), lr=CONFIG['lr'], weight_decay=1e-2)
    sch = optim.lr_scheduler.CosineAnnealingWarmRestarts(opt, T_0=5, T_mult=2, eta_min=1e-6)
    ckpt = os.path.join(CONFIG['ckpt_dir'], f'{label}_s{seed}.pth')
    best_iou = -1.0; noimp = 0
    for ep in range(CONFIG['epochs']):
        model.train()
        for img, mk in tr:
            img = img.to(device); m = mk.to(device).unsqueeze(1).float()
            opt.zero_grad(); out = model(img); loss = loss_fn(out, m); loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0); opt.step()
        sch.step()
        _, viou, _, _, _ = metrics(model, va, loss_fn)
        if viou > best_iou:
            best_iou = viou; noimp = 0; torch.save(model.state_dict(), ckpt)
        else:
            noimp += 1
            if noimp >= CONFIG['patience']: break
    # en iyi ağırlıkla sınama
    model.load_state_dict(torch.load(ckpt, map_location=device))
    te = DataLoader(DepremDataset(files_for('test'), val_tf), batch_size=CONFIG['batch'], shuffle=False, num_workers=2, pin_memory=True)
    _, tiou, tprec, trec, tf1 = metrics(model, te, loss_fn)
    del model, opt, sch
    if device.type=='cuda': torch.cuda.empty_cache()
    return {'model':label,'seed':seed,'val_iou':round(best_iou,4),'test_iou':round(tiou,4),
            'precision':round(tprec,4),'recall':round(trec,4),'f1':round(tf1,4)}

print('Cihaz:', device, '| bileşenler hazır.')

## 6. Protokol döngüsü (resumable) — tohum-dış, model-iç

In [ ]:
import csv
FIELDS = ['model','seed','val_iou','test_iou','precision','recall','f1']

def load_done():
    done = set()
    if os.path.exists(CONFIG['results_csv']):
        with open(CONFIG['results_csv'], newline='') as fh:
            for row in csv.DictReader(fh): done.add((row['model'], int(row['seed'])))
    return done

def append_row(row):
    new = not os.path.exists(CONFIG['results_csv'])
    with open(CONFIG['results_csv'], 'a', newline='') as fh:
        w = csv.DictWriter(fh, fieldnames=FIELDS)
        if new: w.writeheader()
        w.writerow(row)

done = load_done()
total = len(CONFIG['seeds'])*len(MODELS)
print(f'Tamamlanan: {len(done)}/{total}. Kalanlar koşulacak (bittikçe Drive\'a yazılır, kopunca kaldığı yerden).')

for seed in CONFIG['seeds']:            # tohum-dış: ilk tohum bitince 9 modelin tamamı hazır
    for label, arch, encoder in MODELS:
        if (label, seed) in done:
            print(f'⏭️  {label} s{seed} zaten var, atlandı'); continue
        print(f'▶️  {label} · tohum {seed} eğitiliyor...')
        try:
            row = train_one_run(label, arch, encoder, seed)
            append_row(row); done.add((label, seed))
            print(f'   ✅ {row}')
        except Exception as e:
            print(f'   ❌ {label} s{seed} HATA: {e}')
print('Bu oturumdaki koşumlar bitti.')

## 7. Tablo 13 — özet (model başına ortalama ± std)

In [ ]:
import csv, math
rows = []
if os.path.exists(CONFIG['results_csv']):
    with open(CONFIG['results_csv'], newline='') as fh:
        rows = list(csv.DictReader(fh))
from collections import defaultdict
agg = defaultdict(list)
for r in rows: agg[r['model']].append(float(r['test_iou']))
def ms(v):
    n=len(v); mu=sum(v)/n; sd=(sum((x-mu)**2 for x in v)/n)**.5 if n>1 else 0.0
    return mu, sd, n
print(f'{"model":<20}{"test IoU (ort±std)":>24}{"n":>4}')
out = []
for label,_,_ in MODELS:
    if agg[label]:
        mu,sd,n = ms(agg[label]); print(f'{label:<20}{mu:>12.4f} ± {sd:<8.4f}{n:>4}')
        out.append({'model':label,'test_iou_mean':round(mu,4),'test_iou_std':round(sd,4),'n_seeds':n})
    else:
        print(f'{label:<20}{"(henüz koşulmadı)":>24}{0:>4}')
with open(os.path.join(DRIVE_ROOT,'tablo13_ozet.csv'),'w',newline='') as fh:
    w=csv.DictWriter(fh, fieldnames=['model','test_iou_mean','test_iou_std','n_seeds']); w.writeheader(); [w.writerow(o) for o in out]
print('\nÖzet →', os.path.join(DRIVE_ROOT,'tablo13_ozet.csv'))